# Task 7 — ERFNet semantic and anomaly segmentation evaluation

This notebook evaluates ERFNet as the pixel-based baseline. 

The evaluation is divided into two parts.

First, we evaluate the semantic segmentation quality of the pretrained ERFNet model on the Cityscapes validation set. We report:

- the standard mIoU over the 19 Cityscapes trainId classes;
- the mIoU over the 16 classes considered in the EoMT comparison, excluding `pole`, `traffic sign` and `rider`.

Second, we use ERFNet as a pixel-based anomaly segmentation baseline. Since ERFNet produces dense per-pixel class logits, anomaly scores can be computed directly from the logits using post-hoc confidence and uncertainty methods:

- **MaxLogit**: anomaly score = negative maximum class logit;
- **MSP**: anomaly score = 1 - maximum softmax probability;
- **Entropy**: anomaly score = predictive entropy.

The anomaly evaluation is performed on the validation anomaly datasets using AUPRC and FPR@95TPR.

## 1. Environment and paths

This notebook assumes that the repository and large files have already been prepared by the setup notebook.  
Only Google Drive is mounted here, and the project paths are defined.

In [1]:
# Environment prepared by 00_setup: here we only mount Google Drive and define paths.
from google.colab import drive
drive.mount("/content/drive")

import os, sys
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJECT_ROOT = DRIVE_ROOT / "MaskArchitectureAnomaly_CourseProject"
EVAL_DIR = PROJECT_ROOT / "eval"
LARGE_FILES = DRIVE_ROOT / "FAIML_project_and_presentation" / "01_Project" / "large_files"

# Make project modules importable if needed.
for p in (PROJECT_ROOT, PROJECT_ROOT / "eomt", EVAL_DIR):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))


CITYSCAPES_DIR = LARGE_FILES / "datasets" / "cityscapes"

TRAINED_MODELS_DIR = PROJECT_ROOT / "trained_models"
ERFNET_WEIGHTS = TRAINED_MODELS_DIR / "erfnet_pretrained.pth"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("EVAL_DIR:", EVAL_DIR)
print("LARGE_FILES:", LARGE_FILES)
print("CITYSCAPES_DIR:", CITYSCAPES_DIR)
print("TRAINED_MODELS_DIR:", TRAINED_MODELS_DIR)

print("\nChecks:")
print("eval_iou.py exists:", (EVAL_DIR / "eval_iou.py").exists())
print("evalAnomaly.py exists:", (EVAL_DIR / "evalAnomaly.py").exists())
print("ERFNet weights exist:", ERFNET_WEIGHTS.exists())
print("Cityscapes leftImg8bit/val exists:", (CITYSCAPES_DIR / "leftImg8bit" / "val").exists())
print("Cityscapes gtFine/val exists:", (CITYSCAPES_DIR / "gtFine" / "val").exists())

## 1.1 Cityscapes zip files

The Cityscapes validation images and ground-truth annotations are stored on Google Drive as compressed zip files.

To avoid extracting the full dataset inside Google Drive, which would be slow and would occupy unnecessary storage space, the zip files are kept on Drive and extracted only temporarily into the Colab local filesystem.

In particular, the notebook uses:

- `leftImg8bit_trainvaltest.zip`, containing the RGB Cityscapes images;
- `gtFine_trainvaltest.zip`, containing the fine semantic annotations.

The extracted files are stored under `/content/cityscapes`, which is temporary and is automatically cleared when the Colab runtime is reset.

In [2]:
import zipfile
from pathlib import Path

CITYSCAPES_ZIP_DIR = LARGE_FILES / "datasets" / "cityscapes"

LEFT_ZIP = CITYSCAPES_ZIP_DIR / "leftImg8bit_trainvaltest.zip"
GTFINE_ZIP = CITYSCAPES_ZIP_DIR / "gtFine_trainvaltest.zip"

CITYSCAPES_DIR = Path("/content/cityscapes")
CITYSCAPES_DIR.mkdir(parents=True, exist_ok=True)

print("LEFT_ZIP exists:", LEFT_ZIP.exists())
print("GTFINE_ZIP exists:", GTFINE_ZIP.exists())
print("CITYSCAPES_DIR:", CITYSCAPES_DIR)

## 1.2 Temporary extraction of Cityscapes

The following cell extracts the required Cityscapes folders into the temporary Colab storage.

The extraction is performed only if the expected folders are not already present. This avoids extracting the same files multiple times during the same runtime session.


In [3]:
def unzip_if_needed(zip_path, extract_dir, expected_folder):
    expected_path = extract_dir / expected_folder

    if expected_path.exists():
        print(f"{expected_folder} already extracted:", expected_path)
        return

    if not zip_path.exists():
        raise FileNotFoundError(f"Zip file not found: {zip_path}")

    print(f"Extracting {zip_path.name} to {extract_dir}...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_dir)

    print(f"Done extracting {zip_path.name}")


unzip_if_needed(LEFT_ZIP, CITYSCAPES_DIR, "leftImg8bit")
unzip_if_needed(GTFINE_ZIP, CITYSCAPES_DIR, "gtFine")

## 1.3 Cityscapes structure check

After extraction, we verify that the Cityscapes validation split is available in the expected format.

For the semantic mIoU evaluation, ERFNet requires:

- RGB images inside `leftImg8bit/val`;
- semantic labels inside `gtFine/val`;
- labels encoded as `*_labelTrainIds.png`.

The `labelTrainIds` format maps the official Cityscapes label IDs to consecutive training IDs from 0 to 18. This is the format expected by the ERFNet evaluation script.

In [4]:
print("Cityscapes leftImg8bit/val exists:", (CITYSCAPES_DIR / "leftImg8bit" / "val").exists())
print("Cityscapes gtFine/val exists:", (CITYSCAPES_DIR / "gtFine" / "val").exists())

left_images = list((CITYSCAPES_DIR / "leftImg8bit" / "val").rglob("*_leftImg8bit.png"))
label_train_ids = list((CITYSCAPES_DIR / "gtFine" / "val").rglob("*_labelTrainIds.png"))

print("Number of Cityscapes val images:", len(left_images))
print("Number of Cityscapes val labelTrainIds:", len(label_train_ids))

Cityscapes leftImg8bit/val exists: True
Cityscapes gtFine/val exists: True
Number of Cityscapes val images: 500
Number of Cityscapes val labelTrainIds: 0


## 1.4 Generation of Cityscapes `labelTrainIds`

The official Cityscapes annotations are usually provided as polygon annotations and label-ID masks. However, the ERFNet evaluation code expects the semantic ground truth in `labelTrainIds` format.

For this reason, we generate the missing `*_labelTrainIds.png` files using the official Cityscapes preparation script.

This conversion is performed only inside `/content/cityscapes`, so it does not modify or expand the dataset stored on Google Drive.

The conversion maps the Cityscapes semantic classes to consecutive training IDs.

Pixels belonging to ignored or void classes are assigned to the ignore label.

In [6]:
import os
import subprocess
import sys

# Install Cityscapes scripts if not already available
try:
    import cityscapesscripts
    print("cityscapesscripts already installed")
except ImportError:
    print("Installing cityscapesscripts...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "cityscapesscripts"],
        check=True
    )

# The script uses this environment variable to find gtFine
os.environ["CITYSCAPES_DATASET"] = str(CITYSCAPES_DIR)

print("CITYSCAPES_DATASET:", os.environ["CITYSCAPES_DATASET"])

# Generate *_labelTrainIds.png files from Cityscapes annotations
print("Generating labelTrainIds...")
result = subprocess.run(
    [sys.executable, "-m", "cityscapesscripts.preparation.createTrainIdLabelImgs"],
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print("ERROR:")
    print(result.stderr)
else:
    print("labelTrainIds generation completed.")

## 1.5 Final check of generated labels

After generating the `labelTrainIds`, we check again that the number of validation images and semantic labels is consistent.

For the Cityscapes validation split, we expect:

- 500 RGB validation images;
- 500 corresponding `labelTrainIds` masks.

If both counts are equal to 500, the dataset is ready for the ERFNet mIoU evaluation.

In [7]:
label_ids = list((CITYSCAPES_DIR / "gtFine" / "val").rglob("*_labelIds.png"))
polygons = list((CITYSCAPES_DIR / "gtFine" / "val").rglob("*_polygons.json"))
label_train_ids = list((CITYSCAPES_DIR / "gtFine" / "val").rglob("*_labelTrainIds.png"))

print("labelIds:", len(label_ids))
print("polygons:", len(polygons))
print("labelTrainIds:", len(label_train_ids))

print("\nExample files:")
for p in label_ids[:3]:
    print(p)

labelIds: 500
polygons: 500
labelTrainIds: 500

Example files:
/content/cityscapes/gtFine/val/munster/munster_000155_000019_gtFine_labelIds.png
/content/cityscapes/gtFine/val/munster/munster_000003_000019_gtFine_labelIds.png
/content/cityscapes/gtFine/val/munster/munster_000112_000019_gtFine_labelIds.png


## 2. Semantic segmentation evaluation on Cityscapes

Before using ERFNet for anomaly segmentation, we evaluate its semantic segmentation quality on Cityscapes validation images.

We report two metrics:

- **mIoU-19**, computed on all 19 Cityscapes trainId classes;
- **mIoU-16**, computed only on the 16 classes used in the EoMT comparison.

The excluded classes are:

- `pole`;
- `traffic sign`;
- `rider`.

In [8]:
import re
import subprocess
import pandas as pd

miou_cmd = [
    "python", str(EVAL_DIR / "eval_iou.py"),
    "--datadir", str(CITYSCAPES_DIR),
    "--subset", "val",
    "--loadDir", str(TRAINED_MODELS_DIR) + "/",
    "--loadWeights", ERFNET_WEIGHTS.name,
]

print("Running ERFNet semantic evaluation on Cityscapes val...")
print("Command:")
print(" ".join(miou_cmd))

miou_result = subprocess.run(
    miou_cmd,
    capture_output=True,
    text=True
)

print(miou_result.stdout)

if miou_result.returncode != 0:
    print("ERROR:")
    print(miou_result.stderr)

## 3. Semantic segmentation results table

The table below summarizes the ERFNet semantic segmentation performance on the Cityscapes validation set.


In [9]:
MIOU_19_RE = re.compile(r"MEAN IoU on 19 Cityscapes classes:\s*.*?([0-9]+\.[0-9]+)")
MIOU_16_RE = re.compile(r"MEAN IoU on 16 EoMT classes:\s*([0-9]+\.[0-9]+)")

miou_19_match = MIOU_19_RE.search(miou_result.stdout)
miou_16_match = MIOU_16_RE.search(miou_result.stdout)

erfnet_miou_19 = float(miou_19_match.group(1)) if miou_19_match else None
erfnet_miou_16 = float(miou_16_match.group(1)) if miou_16_match else None

semantic_results = pd.DataFrame([
    {
        "model": "ERFNet",
        "dataset": "Cityscapes val",
        "metric": "mIoU-19",
        "value": erfnet_miou_19,
        "excluded_classes": "none",
    },
    {
        "model": "ERFNet",
        "dataset": "Cityscapes val",
        "metric": "mIoU-16",
        "value": erfnet_miou_16,
        "excluded_classes": "pole, traffic sign, rider",
    },
])

semantic_results

,model,dataset,metric,value,excluded_classes
0,ERFNet,Cityscapes val,mIoU-19,72.2,none
1,ERFNet,Cityscapes val,mIoU-16,75.1,"pole, traffic sign, rider"


## 4. Load the anomaly validation datasets

The anomaly validation datasets are stored as a zip file in Google Drive.  
They are extracted temporarily to `/content` so that inference is faster during the Colab session.

In [10]:
from pathlib import Path
import zipfile

ANOMALY_ZIP = LARGE_FILES / "datasets" / "anomaly" / "Anomaly_Validation_Datasets.zip"
LOCAL_DATA_DIR = Path("/content/anomaly_data")

print("Zip exists:", ANOMALY_ZIP.exists())
if not LOCAL_DATA_DIR.exists():
    print("Extracting anomaly datasets temporarily to /content...")
    with zipfile.ZipFile(ANOMALY_ZIP, "r") as z:
        z.extractall(LOCAL_DATA_DIR)
    print("Extraction completed.")
else:
    print("Datasets already extracted in this runtime.")

DATA_ROOT = LOCAL_DATA_DIR / "Validation_Dataset"
TRAINED_MODELS_DIR = PROJECT_ROOT / "trained_models"
ERFNET_WEIGHTS = TRAINED_MODELS_DIR / "erfnet_pretrained.pth"

print("DATA_ROOT exists:", DATA_ROOT.exists())
print("TRAINED_MODELS_DIR exists:", TRAINED_MODELS_DIR.exists())
print("ERFNet weights exist:", ERFNET_WEIGHTS.exists())

## 5. Define datasets and post-hoc methods

The same anomaly validation datasets are used for all methods.  
The image extension is dataset-specific, so each dataset uses its own glob pattern.

In [11]:
import glob

# Dataset image patterns. These names are kept consistent with the project folder names.
datasets = {
    "FS_LostFound_full": DATA_ROOT / "FS_LostFound_full" / "images" / "*.png",
    "fs_static": DATA_ROOT / "fs_static" / "images" / "*.jpg",
    "RoadAnomaly": DATA_ROOT / "RoadAnomaly" / "images" / "*.jpg",
    "RoadAnomaly21": DATA_ROOT / "RoadAnomaly21" / "images" / "*.png",
    "RoadObsticle21": DATA_ROOT / "RoadObsticle21" / "images" / "*.webp",
}

# Pixel-based post-hoc anomaly scoring methods required for Task 7.
methods = ["maxlogit", "msp", "entropy"]

for name, pattern in datasets.items():
    files = glob.glob(str(pattern))
    print(name, len(files), "images")

FS_LostFound_full 100 images
fs_static 30 images
RoadAnomaly 60 images
RoadAnomaly21 10 images
RoadObsticle21 30 images


## 6. Sanity checks

Before running the full evaluation, we check that the evaluation script, ERFNet implementation and pretrained weights are available.


In [12]:
print("Evaluation script exists:", (EVAL_DIR / "evalAnomaly.py").exists())
print("ERFNet implementation exists:", (EVAL_DIR / "erfnet.py").exists())
print("ERFNet pretrained weights exist:", ERFNET_WEIGHTS.exists())

## 7. Run ERFNet anomaly evaluation

For each dataset and each method, the script is launched as a subprocess. The notebook parses the printed metrics and stores them in a structured table.


In [13]:
import re
import subprocess
import pandas as pd
from datetime import datetime

results = []
full_logs = {}

AUPRC_RE = re.compile(r"AUPRC score:\s*([0-9.]+)")
FPR_RE = re.compile(r"FPR@TPR95:\s*([0-9.]+)")

start_time = datetime.now()
print("Evaluation started at:", start_time.strftime("%Y-%m-%d %H:%M:%S"))

for dataset_name, input_pattern in datasets.items():
    for method in methods:
        print(f"\nRunning {method:8s} on {dataset_name}")

        cmd = [
            "python", str(EVAL_DIR / "evalAnomaly.py"),
            "--input", str(input_pattern),
            "--loadDir", str(TRAINED_MODELS_DIR) + "/",
            "--loadWeights", "erfnet_pretrained.pth",
            "--method", method,
        ]

        result = subprocess.run(cmd, capture_output=True, text=True)
        stdout = result.stdout
        stderr = result.stderr
        full_logs[(dataset_name, method)] = {"stdout": stdout, "stderr": stderr}

        if result.returncode != 0:
            print("  ERROR: evaluation script failed.")
            print(stderr)
            continue

        auprc_match = AUPRC_RE.search(stdout)
        fpr_match = FPR_RE.search(stdout)

        if auprc_match is None or fpr_match is None:
            print("  WARNING: metrics could not be parsed from stdout.")
            if stderr:
                print("  STDERR:", stderr[:1000])
            continue

        auprc = float(auprc_match.group(1))
        fpr95 = float(fpr_match.group(1))

        results.append({
            "model": "ERFNet",
            "dataset": dataset_name,
            "method": method,
            "AUPRC": auprc,
            "FPR95": fpr95,
        })

        print(f"  AUPRC = {auprc:.2f} | FPR95 = {fpr95:.2f}")

end_time = datetime.now()
print("\nEvaluation completed at:", end_time.strftime("%Y-%m-%d %H:%M:%S"))
print("Elapsed time:", end_time - start_time)

results_df = pd.DataFrame(results)
results_df

## 8. Anomaly segmentation results tables

The following tables summarize the ERFNet anomaly segmentation results for each anomaly dataset and each post-hoc scoring method.

We report:

- **AUPRC**, where higher values indicate better anomaly detection performance;
- **FPR@95TPR**, where lower values indicate better anomaly detection performance.

In [7]:
# AUPRC table: higher is better.
auprc_table = results_df.pivot(index="dataset", columns="method", values="AUPRC")
auprc_table = auprc_table[["maxlogit", "msp", "entropy"]]
auprc_table["best_method"] = auprc_table[["maxlogit", "msp", "entropy"]].idxmax(axis=1)
auprc_table

method,maxlogit,msp,entropy,best_method
dataset,,,,
FS_LostFound_full,3.300986,1.747985,2.582244,maxlogit
RoadAnomaly,15.581526,12.421447,12.668263,maxlogit
RoadAnomaly21,38.296645,29.083959,30.956045,maxlogit
RoadObsticle21,4.630934,2.713296,3.048639,maxlogit
fs_static,9.499818,7.473543,8.838664,maxlogit


In [8]:
# FPR95 table: lower is better.
fpr95_table = results_df.pivot(index="dataset", columns="method", values="FPR95")
fpr95_table = fpr95_table[["maxlogit", "msp", "entropy"]]
fpr95_table["best_method"] = fpr95_table[["maxlogit", "msp", "entropy"]].idxmin(axis=1)
fpr95_table

method,maxlogit,msp,entropy,best_method
dataset,,,,
FS_LostFound_full,45.532955,50.657505,50.213916,maxlogit
RoadAnomaly,73.279385,82.583275,82.756523,maxlogit
RoadAnomaly21,59.373450,62.562897,62.675714,maxlogit
RoadObsticle21,48.437955,65.180470,65.876601,maxlogit
fs_static,40.302110,41.837639,41.546897,maxlogit


## 9. Save results

All results are saved to disk.

The final Excel file contains separate sheets for:

- semantic mIoU results;
- raw anomaly results;
- AUPRC table;
- FPR@95TPR table.

In [ ]:
PROJECT_PARENT = DRIVE_ROOT / "FAIML_project_and_presentation" / "01_Project"
OUTPUT_DIR = PROJECT_PARENT / "results" / "task7"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
semantic_csv = OUTPUT_DIR / "task7_erfnet_semantic_miou.csv"
semantic_results.to_csv(semantic_csv, index=False)

In [ ]:
PROJECT_PARENT = DRIVE_ROOT / "FAIML_project_and_presentation" / "01_Project"
OUTPUT_DIR = PROJECT_PARENT / "results" / "task7"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

raw_csv = OUTPUT_DIR / "task7_erfnet_raw_results.csv"
auprc_csv = OUTPUT_DIR / "task7_erfnet_auprc_table.csv"
fpr95_csv = OUTPUT_DIR / "task7_erfnet_fpr95_table.csv"
semantic_csv = OUTPUT_DIR / "task7_erfnet_semantic_miou.csv"

results_df.to_csv(raw_csv, index=False)
auprc_table.to_csv(auprc_csv)
fpr95_table.to_csv(fpr95_csv)
semantic_results.to_csv(semantic_csv, index=False)

print("Saved raw results to:", raw_csv)
print("Saved AUPRC table to:", auprc_csv)
print("Saved FPR95 table to:", fpr95_csv)
print("Saved semantic mIoU results to:", semantic_csv)

In [ ]:
excel_path = OUTPUT_DIR / "task7_erfnet_results.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    semantic_results.to_excel(writer, sheet_name="semantic_miou", index=False)
    results_df.to_excel(writer, sheet_name="anomaly_raw_results", index=False)
    auprc_table.to_excel(writer, sheet_name="AUPRC")
    fpr95_table.to_excel(writer, sheet_name="FPR95")

print("Saved Excel results to:", excel_path)